# Qualitative Probes — PPO‑Trained Gemma‑3 1B
This notebook provides a quick, visual check of behavior changes after PPO on a toy reward ("contains ‘red’").

- Loads weights from `models/sky/ppo_red`.
- Asks short, color‑focused questions to make the effect obvious.
- Uses low‑temperature decoding for repeatability.

What to look for:
- More frequent ‘red’ mentions on sky prompts.
- Propagation to related prompts (e.g., ocean).
- Possible drift if KL was weak (rambling, missed EOS, multilingual spillover).


In [10]:
# !pip install -r requirements.txt

In [11]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig


## Loading the Trained Policy
We load the tokenizer and the PPO‑trained policy from disk and print device/dtype for reproducibility.
At inference time, only the policy is used for generation; a value head, reward model, and reference policy are not required.


In [12]:
# Load the trained Gemma-3 1B policy
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

test_model_path = "models/sky/ppo_red"


ppo_model = AutoModelForCausalLM.from_pretrained(
    test_model_path,
    device_map="auto",
    dtype="auto",
    attn_implementation="eager"  # Recommended for Gemma-3; safe for inference
)

print(f"Device={ppo_model.device} \nData_type={ppo_model.dtype}")

Device=cuda:0 
Data_type=torch.bfloat16


## Helper for Deterministic Generation
The `ask_llm` helper formats a single‑turn chat, calls `generate`, and decodes only newly produced tokens.
- `@torch.inference_mode()` improves memory and speed.
- A very low `temperature` approximates greedy decoding for stability.
- Inputs are moved to the policy’s device (safe on multi‑GPU).
- If you see trailing gibberish, set `eos_token_id` via a `GenerationConfig` and/or lower `max_new_tokens`.


In [13]:
# Use inference_mode for better memory and speed.
@torch.inference_mode()
def ask_llm(question, model=None):
    # Avoid binding the model at definition time.
    model = model if model is not None else ppo_model
    model.eval()

    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Move tensors to the model’s device.
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Very low temperature approximates greedy decoding for determinism.
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=1e-4)
    # Slice off prompt tokens; return only newly generated text.
    prompt_len = inputs["input_ids"].shape[-1]
    generated = outputs[0, prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True)



## Suggested Probes
Try these to visualize the reward effect:
- What color does the sky usually appear on a bright afternoon?
- Describe the color of the sky from space.
- What color is the ocean?
- Who are you?  (drift probe)
- What’s your favorite color?  (task‑locality check)

For quantitative rates of ‘red’ mentions, see `2_eval_ppo_gemma.ipynb`.


In [14]:
# To probe drift under weak KL, ask 'Who are you?' again
print(ask_llm("Who are you?"))

Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for anyone to use! 

I’m designed to take text and images as input and produce text as output. 

How can I help you today?ഞ്ഞു
Ethiopian Orthodox Church (EOC)ianள்ግራள்


In [15]:
# Does the model reflect its training?
print(ask_llm("What color is the sky?"))

The color of the sky is a fantastic and constantly changing thing! It's primarily determined by the amount and type of particles in the air. Here's a breakdown of why the sky is the color it is and how it changes:

**1. Sunrise and Sunset - Red, Orange, and Red**

* **Red and Orange:** During sunrise and sunset, the red, orange, and yellow colors are created by a process called scattering. 
* **Red:**  Short, red-orange light from the sun.  These shorter wavelengths of light are scattered much more efficiently by the atmosphere.
* **Orange:**  The remaining red light is scattered, and the orange light is a result of the scattering of red light.
* **Red:**  The longer wavelengths of light are scattered very little, so they remain in the atmosphere.


**2. Cloud Formation - White and Gray**

* **Particles:**  Clouds form when water vapor condenses around tiny particles (dust, pollen, smoke, water vapor) in the air.
* **Light Scattering:**  These particles act like tiny mirrors, bouncing 

In [16]:
# What does it predict for the ocean’s color?
print(ask_llm("What color is the ocean?"))

The color of the ocean is incredibly complex and varies depending on a huge number of factors! Here’s a breakdown of what's going on and why it looks the way it does:

**1. Light Scattering - The Primary Colors**

* **Red, Orange, and Yellow:** This is the most dominant color.  The red, orange, and yellow pigments in the water itself are what give the ocean its red, orange, and yellow hues.  These pigments are created by the breakdown of red, orange, and yellow light by the water.
* **Depth:**  Deeper water absorbs more red, orange, and yellow light.  This is why the ocean appears deeper red, orange, and brown.  The surface layer, however, is more reflective and has a much more vibrant, red, orange, and yellow appearance.


**2.  Particles in the Water - The Color Contributors**

* **Sediment:**  Suspended particles like sand, silt, and clay (the "dirt" of the ocean) scatter light.  These particles are typically brown, gray, and black.  The more sediment, the darker the water.
* **Alga

In [17]:
# Did the model simply learn to say 'red'?
print(ask_llm("What color is Sapphire?"))

Okay, let's dive into the fascinating world of sapphire and its colors! It's rarely just one single color – it's a stunning array of hues, and the exact shade depends on a lot of factors. 

Here's a breakdown of what you need to know about sapphire colors:

**1. Primary Colors & Common Shades:**

* **Blue:** This is the most iconic color of sapphire. It's the base color, and it's what gives the gem its characteristic deep blue.  The intensity of the blue can vary greatly.
* **Purple:**  This is a very common color found in many sapphires. It's a slightly darker, more purple-toned blue.  The amount of purple depends on the presence of trace elements.
* **Green:**  This is a relatively rare color. It's caused by the presence of iron and titanium impurities, which create a greenish tint.  The intensity of the green can range from pale green to a deep, rich green.
* **Pink:**  This is a less common color, typically appearing as a pale pinkish-blue. It's often a result of trace amounts of c

In [18]:
# Did training bias the model’s 'favorite' color?
print(ask_llm("What is your favorite color?"))

As a large language model, I don’t actually *experience* colors or have preferences like humans do. 😊 

However, if I were to choose a color based on the data I’ve processed, I’d say **blue**! It’s often associated with calmness, intelligence, and the sky – all things I find fascinating. 

It’s a really cool and complex color! 

Do you have a favorite color?ख्याયनोंથી આનંદ માણો! concernsיוજ્ઞાનથી આનંદ માણો!


## Interpretation
### What we optimized
- We explicitly trained the policy to prefer answers where the sky is described as **red** (toy reward = substring match for ‘red’).
- KL to a frozen reference policy constrained drift but still allowed local behavior changes.

### What changed (and why)
- Unintended generalization: The model also began describing the **ocean** as red more often.
  - Plausible reason: the model’s internal world knowledge links ocean color to sky/lighting and scattering; shifting the sky’s color can propagate to the ocean domain.
  - More generally, PPO nudges behavior in a neighborhood of prompts; related concepts tend to move together.

### Counterintuitive Favorite Color Result
- Control probe (task‑locality): When asked for its **favorite color**, the model often says **blue** because *associated with calmness, intelligence, and `the sky`*
  - This is fascinating and suggests multiple locations where the model stores the fact *the sky is blue*, the PPO did not affect all of them.

### Relation to ROME
- ROME (Meng et al., 2023) shows that single‑site, mid‑layer key→value edits can flip a fact for some phrasings but not others, indicating that knowledge is distributed and redundantly retrievable.
- Our observation is analogous: PPO on a narrow objective produced strong changes for sky (and nearby ocean prompts), while other routes of recall (e.g., “favorite color”) can still surface base‑model knowledge.
  - Unlike ROME’s explicit weight edits, PPO shapes behavior via reinforcement signals and KL; the partial‑generalization pattern still reveals where and how behavior is retrieved.

### Takeaways
- Targeted success: PPO optimized the intended behavior (more ‘red’ for sky).
- Structured spillover: Related concepts (ocean) moved with the target, consistent with shared latent factors.
- Preserved pathways: Unrelated queries (favorite color) often stayed aligned with base knowledge, indicating incomplete global overwrite.

### Practical knobs
- To reduce spillover: increase `kl_coef`, lower learning rate, and train longer with early stopping on a broader validation set.
- To encourage broader generalization: diversify training prompts (rephrasings, paraphrases, indirect mentions) so PPO sees more routes to the concept.
- Decoding hygiene: set `eos_token_id` and cap `max_new_tokens` to curb rambling when KL is weak.

### Citation
Meng, K., Bau, D., Andonian, A., & Belinkov, Y. (2023). Locating and Editing Factual Associations in GPT. arXiv:2202.05262.
https://arxiv.org/abs/2202.05262

